# DNN Baseline (MFCC Summary Features)

Simple MLP classifier using the same session split (1-4 train, 5 test).
Gender is excluded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score

repo_root = Path.cwd().parents[1]
csv_path = repo_root / "extracted_features" / "mfcc" / "mfcc_features.csv"
csv_path

WindowsPath('f:/Speech-Emotion-Recognition/extracted_features/mfcc/mfcc_features.csv')

In [2]:
df = pd.read_csv(csv_path)

# Safety filter: valid labels only
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape

(7532, 1157)

In [3]:
# Define metadata columns (drop gender explicitly)
metadata_cols = [
    "path",
    "session",
    "method",
    "gender",
    "emotion",
    "n_annotators",
    "agreement",
]

feature_cols = [c for c in df.columns if c not in metadata_cols]
X = df[feature_cols].copy()
y = df["emotion"].copy()

# Drop rows with any missing values in features
mask = X.notna().all(axis=1)
X = X.loc[mask]
y = y.loc[mask]
df = df.loc[mask]

X.shape, y.shape

((7532, 1150), (7532,))

In [4]:
# Session-based split: Sessions 1-4 train, Session 5 test
train_mask = df["session"].isin([1, 2, 3, 4])
test_mask = df["session"].isin([5])

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]


X_train.shape, X_test.shape

((5882, 1150), (1650, 1150))

In [5]:
# MLP baseline (no normalization)
clf = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-2,
    max_iter=50,
    random_state=42,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1:  {f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.4133
Macro F1:  0.2686

Classification report:

              precision    recall  f1-score   support

         ang       0.40      0.56      0.47       170
         exc       0.48      0.20      0.28       299
         fea       0.00      0.00      0.00        10
         fru       0.35      0.38      0.36       381
         hap       1.00      0.01      0.01       143
         neu       0.39      0.70      0.50       384
         sad       0.58      0.47      0.52       245
         sur       0.00      0.00      0.00        18

    accuracy                           0.41      1650
   macro avg       0.40      0.29      0.27      1650
weighted avg       0.47      0.41      0.38      1650



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py

In [6]:
# Cross-validation on the TRAIN split only (sessions 1-4)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_acc = []
cv_f1 = []

for train_idx, val_idx in skf.split(X_train, y_train):
    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_val = y_train.iloc[val_idx]

    model = MLPClassifier(
        hidden_layer_sizes=(512, 256, 128, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate_init=1e-2,
        max_iter=50,
        random_state=42,
    )
    model.fit(X_tr, y_tr)
    y_val_pred = model.predict(X_val)

    cv_acc.append(accuracy_score(y_val, y_val_pred))
    cv_f1.append(f1_score(y_val, y_val_pred, average="macro"))

print(f"CV Accuracy (mean±std): {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"CV Macro F1 (mean±std):  {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")

f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_

CV Accuracy (mean±std): 0.4575 ± 0.0108
CV Macro F1 (mean±std):  0.2629 ± 0.0080


f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


## Normalized MFCC Evaluation

Run this block to evaluate the same MLP baseline using `mfcc_features_normalized.csv`.


In [7]:
# MLP baseline on normalized MFCC features
csv_path_normalized = repo_root / "extracted_features" / "mfcc" / "mfcc_features_normalized.csv"
df_norm = pd.read_csv(csv_path_normalized)

# Safety filter: valid labels only
df_norm = df_norm[(df_norm["emotion"] != "xxx") & (df_norm["agreement"] > 0)].copy()

feature_cols_norm = [c for c in df_norm.columns if c not in metadata_cols]
X_norm = df_norm[feature_cols_norm].copy()
y_norm = df_norm["emotion"].copy()

# Drop rows with any missing values in features
mask_norm = X_norm.notna().all(axis=1)
X_norm = X_norm.loc[mask_norm]
y_norm = y_norm.loc[mask_norm]
df_norm = df_norm.loc[mask_norm]

# Session-based split: Sessions 1-4 train, Session 5 test
train_mask_norm = df_norm["session"].isin([1, 2, 3, 4])
test_mask_norm = df_norm["session"].isin([5])

X_train_norm = X_norm[train_mask_norm]
y_train_norm = y_norm[train_mask_norm]
X_test_norm = X_norm[test_mask_norm]
y_test_norm = y_norm[test_mask_norm]

clf_norm = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-2,
    max_iter=50,
    random_state=42,
)
clf_norm.fit(X_train_norm, y_train_norm)

y_pred_norm = clf_norm.predict(X_test_norm)

acc_norm = accuracy_score(y_test_norm, y_pred_norm)
f1_norm = f1_score(y_test_norm, y_pred_norm, average="macro")

print(f"Dataset: {csv_path_normalized.name}")
print(f"Accuracy: {acc_norm:.4f}")
print(f"Macro F1:  {f1_norm:.4f}")
print("\nClassification report:\n")
print(classification_report(y_test_norm, y_pred_norm))


Dataset: mfcc_features_normalized.csv
Accuracy: 0.3970
Macro F1:  0.2509

Classification report:

              precision    recall  f1-score   support

         ang       0.35      0.55      0.43       170
         exc       0.43      0.21      0.29       299
         fea       0.00      0.00      0.00        10
         fru       0.41      0.18      0.25       381
         hap       0.14      0.02      0.04       143
         neu       0.37      0.83      0.51       384
         sad       0.55      0.44      0.49       245
         sur       0.00      0.00      0.00        18

    accuracy                           0.40      1650
   macro avg       0.28      0.28      0.25      1650
weighted avg       0.39      0.40      0.35      1650



f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\Speech-Emotion-Recognition\.venv\lib\site-packages\sklearn\metrics\_classification.py